# 06 — Research ablation: pipeline WITH vs WITHOUT ESN

Your research question: **does the ESN dynamic engine help** as a buffer/upsampler from VLA (~2 Hz) to high-rate joint commands?

Baselines:
- **esn** — reservoir dynamic engine @ 100 Hz
- **passthrough_zoh** — zero-order hold @ 100 Hz (buffer, no dynamics)
- **passthrough_raw** — no buffer: joint_cmd only at token rate (~2 Hz)

## Run collection (terminals)

```bash
# A) with ESN
python -m s2r.cli deploy -c config/ablation_with_esn.yaml
# stop after ~20–60s

# B) without ESN (ZOH)
python -m s2r.cli deploy -c config/ablation_no_esn_zoh.yaml

# C) without ESN (raw 2Hz)
python -m s2r.cli deploy -c config/ablation_no_esn_raw.yaml
```

Logs go to `data/raw/ablation_*`.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "_lib"))
from bootstrap import setup
ROOT = setup()

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from s2r.experiments.paths import RAW, ensure_experiment_dirs
from s2r.experiments.ablation import compare_runs, save_ablation_report, analyze_run
from s2r.experiments.inspect_data import load_jsonl_rows, extract_series

ensure_experiment_dirs()
candidates = {
    "esn": RAW / "ablation_with_esn",
    "passthrough_zoh": RAW / "ablation_no_esn_zoh",
    "passthrough_raw": RAW / "ablation_no_esn_raw",
}
paths = {k: v for k, v in candidates.items() if v.exists() and any(v.glob("*.jsonl"))}
print("found runs:", paths)
if not paths:
    print("No ablation logs yet — run the deploy commands in the markdown cell first.")

In [ ]:
if paths:
    report = compare_runs(paths)
    out = save_ablation_report(report)
    print("wrote", out)
    print("ranking smoothest→roughest:", report["ranking_smoothest_to_roughest"])
    df = pd.DataFrame([
        {
            "label": r["label"],
            "token_hz": r["rates_hz"]["action_token"],
            "cmd_hz": r["rates_hz"]["joint_cmd"],
            "upsample_ratio": r["upsample_ratio"],
            "jerk_mean": r["cmd_jerk_proxy"].get("mean"),
            "jerk_p95": r["cmd_jerk_proxy"].get("p95"),
            "latency_p95": r["latency_ms"].get("p95"),
            "tracking_err": r["tracking_err_mean"],
        }
        for r in report["runs"].values()
    ])
    display(df)
else:
    df = pd.DataFrame()

In [ ]:
# Overlay joint_cmd trajectories for qualitative comparison
if paths:
    fig, ax = plt.subplots(figsize=(11, 4))
    for label, p in paths.items():
        series = extract_series(load_jsonl_rows(p))
        cmds = series["cmds"]
        if cmds.size:
            ax.plot(cmds[:800, 0], label=f"{label} (hz≈{series['cmd_hz']:.1f})")
    ax.set_title("joint_cmd[0] — ESN vs no-ESN baselines")
    ax.legend(); ax.set_xlabel("sample"); ax.set_ylabel("q0")
    plt.tight_layout(); plt.show()

## What to claim in a research writeup

- **Rate recovery:** ESN/ZOH should show `cmd_hz >> token_hz`; raw should not.
- **Smoothness:** ESN should reduce jerk vs raw (and often vs ZOH) if trained/bootstrapped well.
- **Tracking:** compare state vs command error under the same plant (`state_publisher`).
- **Latency:** ESN inference cost should stay tiny vs VLA/VLM.

CLI shortcut after logs exist:

```bash
python -m s2r.cli compare-ablation
```